# Imports and Initialization

In [1]:
import qlib
from qlib.config import REG_US # [REG_CN, REG_US]

from qlib.utils import init_instance_by_config
from qlib.utils import flatten_dict
from qlib.workflow import R


import pandas as pd
import numpy as np

import torch
from torch.backends import cuda

import sys
import warnings


# Add custom library to path
sys.path.append("C:\Projects\Investing\QuantitativeML_Investing\_Lib")


# Init Qlib
provider_uri = "C:\Projects\Investing\QuantitativeML_Investing\Database\Qdata/SPAIN"  # target_dir
qlib.init(provider_uri=provider_uri, region=REG_US)
print("Qlib version: " + str(qlib.__version__))


# GPU
print("GPU enabled: " + str(torch.cuda.is_available()) + "\n")


# Warnings
warnings.filterwarnings("ignore")

[64372:MainThread](2022-09-11 14:17:06,291) INFO - qlib.Initialization - [config.py:413] - default_conf: client.
[64372:MainThread](2022-09-11 14:17:06,299) INFO - qlib.Initialization - [__init__.py:74] - qlib successfully initialized based on client settings.
[64372:MainThread](2022-09-11 14:17:06,300) INFO - qlib.Initialization - [__init__.py:76] - data_path={'__DEFAULT_FREQ': WindowsPath('C:/Projects/Investing/QuantitativeML_Investing/Database/Qdata/SPAIN')}


Qlib version: 0.8.6
GPU enabled: True



In [2]:
instruments = 'all'
benchmark = '^ibex'

# Data and Model

## Data Loader

In [3]:
###################################
# Define DataLoader
###################################
from qlib.data.dataset.loader import QlibDataLoader
from furlag_qlib import qdatasets, qfilters  # Custom Class


##################################################################################################################
# --------------
# Features
# --------------

fields, names = qdatasets.dataset_features_ts_()

##################################################################################################################
# --------------
# Labels
# --------------

labels = ['''((Ref($open, -2) / Ref($open, -1)) - 1)''']
# labels = ['''((EMA(Ref($open, -5), 5) / Ref($open, -1)) - 1)''']
# labels = ['''((EMA(Ref($open, -5), 5) / EMA($open, 5)) - 1)''']

label_names = ['label']

##################################################################################################################
# --------------
# Filters
# --------------

filters = qfilters.Filters()

# Filter pipe
filter_pipe = filters.basic + filters.spain


##################################################################################################################
##################################################################################################################
# ----------------------------
# Initialize data loader
# ----------------------------

data_loader = QlibDataLoader(config={"feature": (fields, names), "label": (labels, label_names)},
                             filter_pipe=filter_pipe, freq="day")

print(data_loader)
print("Number of features == " + str(len(fields)))

Number of features == 197


## Data Handler

In [4]:
###################################
# Define DataHandler
###################################
from qlib.data.dataset.handler import DataHandler, DataHandlerLP
from qlib.data.dataset.processor import ProcessInf, RobustZScoreNorm, Fillna, DropnaProcessor, DropnaLabel, CSZScoreNorm

from furlag_qlib.qprocessors import DropLabelOutliners  # Custom Class


defined_datahandler = DataHandlerLP(instruments=instruments,
                                    # start_time='2000-01-01',
                                    # end_time='2022-01-01',
                                    data_loader=data_loader,
                                    infer_processors=[
                                            # ProcessInf(),
                                            RobustZScoreNorm(
                                                fields_group="feature",
                                                fit_start_time="2000-01-01", fit_end_time="2018-12-31",
                                                clip_outlier="False"
                                            ),
                                            Fillna(fields_group="feature", fill_value=0),
                                            # DropnaProcessor(fields_group="feature")
                                       ],
                                    learn_processors=[
                                            DropnaLabel(),
                                            # DropLabelOutliners(0.2),
                                            CSZScoreNorm(fields_group="label")
                                       ]
                                    )

[64372:MainThread](2022-09-11 14:17:58,032) INFO - qlib.timer - [log.py:117] - Time cost: 50.373s | Loading data Done
[64372:MainThread](2022-09-11 14:18:01,074) INFO - qlib.timer - [log.py:117] - Time cost: 2.819s | RobustZScoreNorm Done
[64372:MainThread](2022-09-11 14:18:01,175) INFO - qlib.timer - [log.py:117] - Time cost: 0.099s | Fillna Done
[64372:MainThread](2022-09-11 14:18:01,332) INFO - qlib.timer - [log.py:117] - Time cost: 0.102s | DropnaLabel Done
[64372:MainThread](2022-09-11 14:18:12,200) INFO - qlib.timer - [log.py:117] - Time cost: 10.866s | CSZScoreNorm Done
[64372:MainThread](2022-09-11 14:18:12,214) INFO - qlib.timer - [log.py:117] - Time cost: 14.181s | fit & process data Done
[64372:MainThread](2022-09-11 14:18:12,216) INFO - qlib.timer - [log.py:117] - Time cost: 64.557s | Init data Done


## Task Definition

In [5]:
###################################
# Define task
###################################

task = {
    "model": {
            "class": "ALSTM",
            "module_path": "furlag_qlib.model.pytorch_alstm_ts",
            "kwargs": {
                'd_feat': len(fields),
                'early_stop': 10,
                'n_epochs': 200,
                
                'batch_size': 2048,  # 2048
                'hidden_size': 128,  # 128
                'num_layers': 2,  # 2
                'dropout': 0,  # 0
                'lr': 0.0001,  # 0.0001
                'rnn_type': 'GRU',  # 'GRU', 'LSTM'
                
                'metric': 'loss',
                'loss': 'mse',

                'seed': 42,

                'n_jobs': 0,
                'GPU': 0
            },
    },
    "dataset": {
        "class": "TSDatasetH",
        "module_path": "qlib.data.dataset",
        "kwargs": {
            "step_len": 20,
            "handler": defined_datahandler,
            "segments": {
                "train": ("2000-01-01", "2018-12-31"),
                "valid": ("2019-01-01", "2021-12-31"),
                "test": ("2019-01-01", "2021-12-31"),
            },
        },
    },
}


###################################
# Initialize dataset
###################################

dataset = init_instance_by_config(task["dataset"])

print(dataset)


###################################
# Initialize model
###################################

model = init_instance_by_config(task["model"])

print(model)

[64372:MainThread](2022-09-11 14:18:12,313) INFO - qlib.ALSTM - [pytorch_alstm_ts.py:62] - ALSTM pytorch version...
[64372:MainThread](2022-09-11 14:18:12,319) INFO - qlib.ALSTM - [pytorch_alstm_ts.py:110] - ALSTM parameters setting:
d_feat : 197
hidden_size : 128
num_layers : 2
dropout : 0
n_epochs : 200
lr : 0.0001
metric : loss
batch_size : 2048
early_stop : 10
optimizer : adam
loss_type : mse
device : cuda:0
n_jobs : 0
use_GPU : True
seed : 42
[64372:MainThread](2022-09-11 14:18:12,334) INFO - qlib.ALSTM - [pytorch_alstm_ts.py:129] - model:
ALSTMModel(
  (net): Sequential(
    (fc_in): Linear(in_features=197, out_features=128, bias=True)
    (act): Tanh()
  )
  (rnn): GRU(128, 128, num_layers=2, batch_first=True)
  (fc_out): Linear(in_features=256, out_features=1, bias=True)
  (att_net): Sequential(
    (att_fc_in): Linear(in_features=128, out_features=64, bias=True)
    (att_dropout): Dropout(p=0, inplace=False)
    (att_act): Tanh()
    (att_fc_out): Linear(in_features=64, out_fe

TSDatasetH(handler=<qlib.data.dataset.handler.DataHandlerLP object at 0x0000025A86A860C8>, segments={'train': ('2000-01-01', '2018-12-31'), 'valid': ('2019-01-01', '2021-12-31'), 'test': ('2019-01-01', '2021-12-31')})


## Model Training/Loading

In [6]:
###################################
# Model
###################################

train = True

if train:
    # Train model
    with R.start(experiment_name='train', recorder_name='train_recorder'):
        R.log_params(**flatten_dict(task))

        # model.fit(dataset)
        model.train_n_epochs(dataset, n_epochs=10)
        
        R.save_objects(trained_model=model)
        rid = R.get_recorder().id

else:
    # Load model
    recorder = R.get_recorder(experiment_name='train', recorder_name='train_recorder')
    model = recorder.load_object("trained_model")

[64372:MainThread](2022-09-11 14:18:17,386) INFO - qlib.workflow - [expm.py:315] - <mlflow.tracking.client.MlflowClient object at 0x0000025A97D243C8>
[64372:MainThread](2022-09-11 14:18:17,397) INFO - qlib.workflow - [exp.py:257] - Experiment 1 starts running ...
[64372:MainThread](2022-09-11 14:18:17,891) INFO - qlib.workflow - [recorder.py:293] - Recorder 4552a0772a394fd3ae4133af4771554e starts running under Experiment 1 ...
[64372:MainThread](2022-09-11 14:18:19,220) INFO - qlib.ALSTM - [pytorch_alstm_ts.py:314] - Epoch0:
[64372:MainThread](2022-09-11 14:18:19,221) INFO - qlib.ALSTM - [pytorch_alstm_ts.py:315] - training...
[64372:MainThread](2022-09-11 14:18:42,025) INFO - qlib.ALSTM - [pytorch_alstm_ts.py:317] - evaluating...
[64372:MainThread](2022-09-11 14:19:07,702) INFO - qlib.ALSTM - [pytorch_alstm_ts.py:320] - train -0.968291, valid -0.976264
[64372:MainThread](2022-09-11 14:19:07,708) INFO - qlib.ALSTM - [pytorch_alstm_ts.py:314] - Epoch1:
[64372:MainThread](2022-09-11 14:1

# Backtest

## Port Analysis Config

In [7]:
###################################
# Define port analysis config
###################################
from furlag_qlib import strategies  # Custom Class
from furlag_qlib.stop_loss import StopLossLogics  # Custom Class


# StopLoss logics (for Strategy)
sll = StopLossLogics()
atr_sll = sll.atr_stoploss(diff=1.0, of='$close', r=14)


# ------------------------------------------------------------


port_analysis_config = {
    "strategy": {
        "class": "KellyCriterionStrategy",
        "module_path": "furlag_qlib.strategies",
        "kwargs": {
            "signal": (model, dataset),

            "max_invest": 0.25,
            "min_invest": 250,
            
            "stop_loss_logics": [atr_sll],
            "hard_stop_loss": False,
            "soft_stop_loss": False,
            "trailing_stop_loss": False,

            "stock_validation": False,
        },
    },

    "backtest": {
        "account": 10000,
        "benchmark": benchmark,

        "start_time": None,
        "end_time": None,

        "exchange_kwargs": {
                "freq": "day",
                "deal_price": "open",
                "open_cost": 0.0005,
                "close_cost": 0.0005,
                "min_cost": 2,
        },
    },

    "executor": {
        "class": "SimulatorExecutor",
        "module_path": "qlib.backtest.executor",
        "kwargs": {
            "time_per_step": "day",
            "generate_portfolio_metrics": True,
        },
    }

}

## Loop Backtest

In [8]:
###################################
# Loop Testing
###################################

from qlib.workflow.record_temp import SignalRecord, PortAnaRecord
from IPython.display import clear_output


info_df = pd.DataFrame()


# backtest and analysis
with R.start(experiment_name='test', recorder_name='test_recorder'):
    recorder = R.get_recorder()
    ba_rid = recorder.id

    for i in range(0, 10):
        print("#####################################################################")
        print("Epoch " + str(i) + ":")
        model.load_model_at_epoch(i)


        # prediction
        sr = SignalRecord(model, dataset, recorder)
        sr.generate()

        # backtest & analysis
        par = PortAnaRecord(recorder, port_analysis_config, "day")
        par.generate()


        analysis_df = recorder.load_object("portfolio_analysis/port_analysis_1day.pkl")


        info_df = info_df.append(analysis_df.transpose())



clear_output(wait=True)



# Display
info_df.reset_index(drop=True, inplace=True)

info_df.to_csv("results.csv")

with pd.option_context('display.max_rows', 100, 'display.max_columns', 100):  # more options can be specified
    display(info_df)

excess_return_without_cost                                                \
                        mean       std annualized_return information_ratio   
0                   0.000265  0.009622          0.063120          0.425201   
1                   0.000690  0.009347          0.164250          1.139072   
2                   0.000447  0.010902          0.106327          0.632167   
3                   0.001024  0.009893          0.243804          1.597477   
4                   0.000339  0.009620          0.080727          0.543973   
5                   0.001012  0.009925          0.240758          1.572410   
6                   0.000867  0.010114          0.206314          1.322316   
7                   0.000718  0.009691          0.170885          1.143013   
8                   0.001108  0.010674          0.263803          1.601944   
9                   0.000823  0.010839          0.195826          1.171076   

               excess_return_with_cost                              \
  max_drawdown                    mean       std annualized_return   
0    -0.207112                0.000116  0.009630          0.027495   
1    -0.194105                0.000518  0.009339          0.123392   
2    -0.178617                0.000301  0.010910          0.071601   
3    -0.160819                0.000880  0.009891          0.209509   
4    -0.146101                0.000179  0.009639          0.042501   
5    -0.161314                0.000872  0.009931          0.207461   
6    -0.124077                0.000647  0.010131          0.153977   
7    -0.142157                0.000474  0.009691          0.112751   
8    -0.141779                0.000923  0.010683          0.219687   
9    -0.143273                0.000591  0.010838          0.140648   

                                  
  information_ratio max_drawdown  
0          0.185073    -0.237083  
1          0.856422    -0.198650  
2          0.425424    -0.188257  
3          1.372985    -0.163402  
4          0.285815    -0.151295  
5          1.354171    -0.177365  
6          0.985179    -0.137057  
7          0.754184    -0.152258  
8          1.332951    -0.145713  
9          0.841190    -0.150990

## Prediction & Analysis

In [ ]:
###################################
# Load model at n epoch of training
###################################

model.load_model_at_epoch(0)

In [ ]:
###################################
# prediction, backtest & analysis
###################################
from qlib.workflow.record_temp import SignalRecord, PortAnaRecord


with R.start(experiment_name='test', recorder_name='test_recorder'):
    # recorder = R.get_recorder(recorder_id=rid, experiment_name="train")
    # model = recorder.load_object("trained_model")

    # prediction
    recorder = R.get_recorder()
    ba_rid = recorder.id
    sr = SignalRecord(model, dataset, recorder)
    sr.generate()

    # backtest & analysis
    par = PortAnaRecord(recorder, port_analysis_config, "day")
    par.generate()

In [ ]:
from qlib.contrib.report import analysis_model, analysis_position
from qlib.data import D

recorder = R.get_recorder(recorder_id=ba_rid, experiment_name='test')


pred_df = recorder.load_object("pred.pkl")

pred_df_dates = pred_df.index.get_level_values(level='datetime')


report_normal_df = recorder.load_object("portfolio_analysis/report_normal_1day.pkl")
positions = recorder.load_object("portfolio_analysis/positions_normal_1day.pkl")
analysis_df = recorder.load_object("portfolio_analysis/port_analysis_1day.pkl")

## Analysis Position

### Report

In [ ]:
analysis_position.report_graph(report_normal_df)

### Risk Analysis

In [ ]:
analysis_position.risk_analysis_graph(analysis_df, report_normal_df)

### Save

In [ ]:
report_normal_df.to_csv("Analysis/report_normal.csv")
analysis_df.to_csv("Analysis/port_analysis.csv")

In [ ]:
pred_df.to_csv("Analysis/predictions.csv")

In [ ]:
# Save positions
positions_df = pd.DataFrame.from_dict(positions, orient='index')
positions_df.to_csv("Analysis/positions.csv")